In [26]:
# install packages into this notebook's .venv (uv envs have no pip, so use %uv)
%uv pip install langgraph google-genai python-dotenv

/Users/jinyuan/Documents/ClaudeProject/AAI520-Group-7/.venv/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [27]:
from google import genai
from google.genai import types
from pydantic import BaseModel
from dotenv import load_dotenv

load_dotenv()
client = genai.Client()   # reads GEMINI_API_KEY from .env
MODEL = "gemini-3.8-flash"

# 1. Tools declaration

# 2. Agent Declaration

In [28]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END

# create state for multi-step workflow
class State(TypedDict):
    ticker: str
    plan: List[str]
    raw_news: list
    routed: dict
    analysis: str
    critique: str
    score: float
    iterations: int

class Plan(BaseModel):
    steps: List[str]

# Entry point
def planner(state: State) -> dict:
    ticker = state["ticker"].upper()
    response = client.models.generate_content(
        model=MODEL,
        contents=(
            f"Create a 4-6 step research plan to analyze the stock {ticker}. "
            "Cover data gathering (news, earnings, price), analysis, and self-critique. "
            "Keep each step short and actionable."
        ),
        config=types.GenerateContentConfig(
            system_instruction="You are an investment research agent that plans before acting.",
            response_mime_type="application/json",
            response_schema=Plan,
        ),
    )
    if response.parsed is None:
        raise RuntimeError(f"Planner returned no valid plan: {response.text}")

    return {"ticker": ticker, "plan": response.parsed.steps, "iterations": 0}

#
# def fetch(s):
#
# # prompt chain: clean -> classify -> extract -> summarize
# def chain(s):
#
#
# def save_memory(s):
#
#
# def router(s):


In [30]:
g = StateGraph(State)
# for name, fn in [("planner", planner),
#                  ("fetch", fetch),
#                  ("memory", save_memory)]:
#     g.add_node(name, fn)

g.add_node("planner", planner)
g.set_entry_point("planner")
g.add_edge("planner", END)
# g.add_edge("planner", "fetch")
# g.add_edge("fetch", "chain")
# g.add_conditional_edges("chain", router,
#                         {"earnings": "earnings", "news": "news", "market": "market"})

# g.add_edge("memory", END)


app = g.compile()
out = app.invoke({"ticker": "aapl"})
for i, step in enumerate(out["plan"], 1):
    print(i, step)

1 Retrieve historical AAPL price action, trading volumes, and key technical indicators.
2 Collect latest earnings reports, financial statements, and valuation multiples for AAPL.
3 Gather recent news, analyst revisions, and macroeconomic factors impacting Apple.
4 Conduct fundamental and quantitative analysis to evaluate growth, margins, and valuation.
5 Formulate an investment thesis balancing bull and bear scenarios.
6 Perform a self-critique to identify potential biases, data gaps, and downside risks.
